# M12 · Notebook 05 · Procesos batch: la mitad de precio si puedes esperar

Muchísimo trabajo con LLMs **no necesita respuesta inmediata**: clasificar el histórico de tickets, etiquetar miles de reseñas, generar datos sintéticos, ejecutar una evaluación sobre 2.000 casos, calcular embeddings de un corpus...

Para eso existe la **Batch API**: subes un fichero con todas las peticiones, el proveedor las procesa cuando tiene capacidad libre (máximo 24 h, normalmente minutos) y te devuelve un fichero con los resultados. A cambio: **50% de descuento** en input y output, y límites de rate separados de tu tráfico en tiempo real.

| | Síncrono | Flex (`service_tier="flex"`) | Batch API |
|---|---|---|---|
| Precio | 100% | 50% | 50% |
| Respuesta | Inmediata | Inmediata pero más lenta, puede devolver 429 | Asíncrona, hasta 24 h |
| Interfaz | Una llamada | Una llamada | Fichero JSONL de hasta 50.000 peticiones |
| Ideal para | Usuario esperando | Jobs internos que quieren código simple | Volumen grande y offline |

Precios por millón de tokens ([pricing oficial](https://developers.openai.com/api/docs/pricing)):

| Modelo | Input síncrono | Input batch | Output síncrono | Output batch |
|---|---|---|---|---|
| gpt-6-luna | $0.10 | $0.05 | $0.50 | $0.25 |
| gpt-6-sol | $2.00 | $1.00 | $10.00 | $5.00 |

> El descuento **se combina con el prompt caching** del notebook 04: el input cacheado en batch cuesta $0.005/M en luna. Ojo: en este lab el *system prompt* es corto (menos de 1.024 tokens), así que no llega a cachear; el ejercicio 4 lo alarga para verlo.

## El caso: clasificar reseñas de producto

Tenemos reseñas de clientes y queremos, para cada una, el sentimiento, una puntuación de 1 a 5, los temas que menciona y si requiere que alguien de soporte la conteste. Formato de salida estricto con **Structured Outputs**.

## Setup

In [ ]:
%pip install -qU openai pandas

In [ ]:
import os, getpass, json, time
import pandas as pd
from openai import OpenAI

# En local la clave se lee de la variable de entorno; en Colab se pide por pantalla.
api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("Enter your OpenAI API Key: ")
client = OpenAI(api_key=api_key)

MODEL = "gpt-6-luna"
PRICES = {  # USD por millón de tokens (octubre 2026)
    "sync":  {"input": 0.10, "cached": 0.01, "write": 0.125, "output": 0.50},
    "batch": {"input": 0.05, "cached": 0.005, "write": 0.0625, "output": 0.25},
}

## 1 · Los datos

In [ ]:
RESEÑAS = [
    "Llegó en dos días y funciona perfecto. La batería dura muchísimo, muy contento.",
    "El producto está bien pero el embalaje venía destrozado y faltaba el cargador.",
    "Tercera vez que escribo y nadie me responde. Quiero mi dinero ya.",
    "Calidad precio insuperable. Lo recomiendo a todo el mundo.",
    "La talla no corresponde con la guía, me queda enorme. ¿Cómo hago la devolución?",
    "Funciona, sin más. Esperaba algo mejor por lo que cuesta.",
    "Se rompió la bisagra a las tres semanas. Muy decepcionado con la marca.",
    "Atención al cliente de diez: me cambiaron la unidad defectuosa en 48 h.",
    "Bonito diseño, pero la app para configurarlo se cuelga cada dos por tres.",
    "No ha llegado todavía y el seguimiento lleva una semana sin moverse.",
    "Perfecto para la cocina, fácil de limpiar y muy silencioso.",
    "Me cobraron dos veces el mismo pedido. Necesito que lo revisen urgente.",
    "El color es distinto al de las fotos, pero me gusta igualmente.",
    "Instrucciones solo en inglés y bastante confusas. El producto en sí, bien.",
    "Lo uso a diario desde hace un año y sigue como el primer día.",
    "Hace un ruido extraño al encenderlo. ¿Es normal o está defectuoso?",
    "Envío rapidísimo y el regalo encantó a mi madre.",
    "Pésimo. Dejó de cargar al segundo día y la garantía me pide mil papeles.",
    "Correcto. Cumple lo que promete, ni más ni menos.",
    "La suscripción Premium vale la pena solo por los envíos gratis.",
    "Pedí azul y me llegó negro. Ya es la segunda vez que pasa.",
    "Muy buena relación calidad-precio, aunque el cable es algo corto.",
    "Me encanta, pero el precio ha subido 15 € desde que lo compré.",
    "No puedo descargar la factura desde el área de cliente, da error.",
]
print(len(RESEÑAS), "reseñas")

## 2 · Construir el fichero JSONL

Cada línea es **una petición completa** a `/v1/chat/completions` con un `custom_id` único. Los resultados pueden volver **desordenados**: el `custom_id` es lo que nos permite reunirlos con la entrada.

In [ ]:
TEMAS = ["envío", "producto", "calidad", "precio", "devolución", "garantía", "atención", "pago", "app", "cuenta"]

SYSTEM = f"""Eres un analista de experiencia de cliente de una tienda online.
Clasifica la reseña del usuario. Temas posibles: {", ".join(TEMAS)}.
requiere_respuesta = true si el cliente pide algo, reclama dinero o reporta un problema sin resolver."""

SCHEMA = {
    "name": "clasificacion_resena",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "sentimiento": {"type": "string", "enum": ["positivo", "neutro", "negativo"]},
            "puntuacion": {"type": "integer", "description": "1 (muy mala) a 5 (excelente)"},
            "temas": {"type": "array", "items": {"type": "string", "enum": TEMAS}},  # solo temas de la lista
            "requiere_respuesta": {"type": "boolean"},
        },
        "required": ["sentimiento", "puntuacion", "temas", "requiere_respuesta"],
        "additionalProperties": False,
    },
}

def body(reseña):
    return {
        "model": MODEL,
        "reasoning_effort": "none",
        "messages": [{"role": "system", "content": SYSTEM},
                     {"role": "user", "content": reseña}],
        "response_format": {"type": "json_schema", "json_schema": SCHEMA},
    }

with open("batch_input.jsonl", "w", encoding="utf-8") as f:
    for i, r in enumerate(RESEÑAS):
        f.write(json.dumps({"custom_id": f"resena-{i:03d}", "method": "POST",
                            "url": "/v1/chat/completions", "body": body(r)}, ensure_ascii=False) + "\n")

print(open("batch_input.jsonl", encoding="utf-8").readline()[:400], "...")

## 3 · Subir el fichero y lanzar el batch

In [ ]:
batch_file = client.files.create(file=open("batch_input.jsonl", "rb"), purpose="batch")

batch = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",            # único valor admitido
    metadata={"lab": "M12-NB05", "descripcion": "clasificacion de reseñas"},
)
BATCH_ID = batch.id
print("Batch creado:", BATCH_ID, "· estado:", batch.status)

## 4 · Esperar

Estados posibles: `validating` → `in_progress` → `finalizing` → `completed` (o `failed`, `expired`, `cancelled`).

Un batch pequeño suele terminar en pocos minutos, pero **el contrato es de 24 h**. En producción no se hace polling en un bucle: se guarda el `BATCH_ID` y un job programado (o un webhook) recoge el resultado. Si esta celda agota su tiempo, vuelve a ejecutarla más tarde: el batch sigue en marcha.

In [ ]:
def esperar(batch_id, timeout_s=30 * 60, cada_s=15):
    t0 = time.time()
    while True:
        b = client.batches.retrieve(batch_id)
        rc = b.request_counts
        print(f"[{time.time() - t0:5.0f}s] {b.status:<11} completadas={rc.completed} fallidas={rc.failed} total={rc.total}")
        if b.status in ("completed", "failed", "expired", "cancelled"):
            return b
        if time.time() - t0 > timeout_s:
            print("Sigue en marcha. Vuelve a ejecutar esta celda más tarde.")
            return b
        time.sleep(cada_s)

batch = esperar(BATCH_ID)

## 5 · Descargar y unir resultados

In [ ]:
assert batch.status == "completed", f"El batch está en estado {batch.status}"

salida = client.files.content(batch.output_file_id).text
resultados = [json.loads(l) for l in salida.splitlines() if l.strip()]

if batch.error_file_id:  # peticiones individuales que fallaron
    print(client.files.content(batch.error_file_id).text[:1000])

filas, usages, saltadas = [], [], []
for r in resultados:
    # Una petición puede fallar sin que falle el batch: error propio o status_code distinto de 200
    if r.get("error") or (r.get("response") or {}).get("status_code") != 200:
        saltadas.append(r["custom_id"])
        continue
    i = int(r["custom_id"].split("-")[1])
    resp = r["response"]["body"]
    usages.append(resp["usage"])
    datos = json.loads(resp["choices"][0]["message"]["content"])
    filas.append({"id": r["custom_id"], "reseña": RESEÑAS[i][:60], **datos})

if saltadas:
    print(f"Saltadas {len(saltadas)} peticiones con error:", saltadas)

df = pd.DataFrame(filas).sort_values("id").reset_index(drop=True)
df

In [ ]:
print(df.sentimiento.value_counts(), "\n")
print("Requieren respuesta de soporte:", df.requiere_respuesta.sum(), "de", len(df))
df.explode("temas").temas.value_counts()

## 6 · ¿Cuánto nos hemos ahorrado?

In [ ]:
def coste(usages, tarifa):
    p = PRICES[tarifa]
    total = 0
    for u in usages:
        d = u.get("prompt_tokens_details") or {}
        cached, written = d.get("cached_tokens", 0) or 0, d.get("cache_write_tokens", 0) or 0
        normal = u["prompt_tokens"] - cached - written
        total += normal * p["input"] + cached * p["cached"] + written * p["write"] + u["completion_tokens"] * p["output"]
    return total / 1e6

c_batch, c_sync = coste(usages, "batch"), coste(usages, "sync")
print(f"Coste batch:            ${c_batch:.6f}")
print(f"Mismo trabajo síncrono: ${c_sync:.6f}")
print(f"Ahorro: {1 - c_batch / c_sync:.0%}")
print(f"\nProyección a 1 millón de reseñas: batch ${c_batch / len(usages) * 1e6:,.2f} vs síncrono ${c_sync / len(usages) * 1e6:,.2f}")

## 7 · Alternativa sin ficheros: Flex processing

Si no quieres gestionar ficheros ni esperar, `service_tier="flex"` da el **mismo precio que batch** en una llamada normal. A cambio la respuesta puede tardar más y, si no hay capacidad, devuelve un error 429 que hay que reintentar. Es útil para jobs internos con poco volumen.

In [ ]:
t0 = time.time()
r = client.chat.completions.create(**body(RESEÑAS[2]), service_tier="flex", timeout=600)
print(f"{time.time() - t0:.1f}s · tier={r.service_tier}")
print(r.choices[0].message.content)

## Cuándo usar cada cosa

- **Usuario esperando** → síncrono (y prompt caching).
- **Volumen grande, sin prisa**: evaluaciones, etiquetado, histórico, datos sintéticos, embeddings → **Batch API**.
- **Pocas peticiones internas, sin prisa** → Flex.

> Anthropic (Message Batches) y Gemini (Batch Mode) ofrecen el mismo 50% con un flujo casi idéntico: fichero o lista de peticiones, `custom_id`, polling del estado y descarga de resultados.

## Ejercicio

1. Añade 10 reseñas propias y vuelve a lanzar el batch.
2. Añade un campo `idioma` al schema y prueba con reseñas en inglés y catalán.
3. Calcula cuánto costaría clasificar 1 millón de reseñas con `gpt-6-sol` en batch frente a síncrono.
4. Aquí el `SYSTEM` es corto (unos 200 tokens de prompt por petición, por debajo del mínimo de 1.024), así que **no cachea**: `cached_tokens` sale a 0. Alárgalo por encima de 1.024 tokens (por ejemplo, con una guía detallada de cada tema y ejemplos de reseñas ya clasificadas), relanza el batch y comprueba `cached_tokens` en `usages`. ¿Cuánto baja el coste del input?

## Limpieza

In [ ]:
# Opcional: borrar los ficheros subidos (los de salida se borran solos a los 30 días)
client.files.delete(batch_file.id)
os.remove("batch_input.jsonl")